In [ ]:
# =========================================================
# FORECASTING DATA PREPROCESSING PIPELINE
# Google Colab Compatible
# Input  : rideBookings.csv
# Output : rideBookings_preprocessed.csv
# =========================================================

# =========================
# IMPORT LIBRARIES
# =========================
import pandas as pd
import numpy as np

# =========================
# UPLOAD FILE IN COLAB
# =========================
# from google.colab import files

# uploaded = files.upload()

# # Automatically get uploaded filename
# input_file = list(uploaded.keys())[0]

# =========================
# LOAD DATA
# =========================
df = pd.read_csv('/content/rideBookings.csv')

print("Dataset Shape:", df.shape)

# =========================================================
# DO NOT TOUCH FIRST 4 COLUMNS
# =========================================================

protected_cols = df.columns[:4]

print("\nProtected Columns:")
print(protected_cols.tolist())

# =========================================================
# BASIC CLEANING
# =========================================================

# Remove unnecessary quotes/spaces
for col in df.select_dtypes(include='object').columns:

    df[col] = (
        df[col]
        .astype(str)
        .str.replace('"', '', regex=False)
        .str.strip()
    )

# Replace string nulls
null_strings = ['nan', 'NaN', 'None', 'null', 'NULL', '']

df.replace(null_strings, np.nan, inplace=True)

# =========================================================
# DATETIME FEATURE ENGINEERING
# =========================================================

try:

    df['Datetime'] = pd.to_datetime(
        df['Date'] + ' ' + df['Time'],
        errors='coerce'
    )

    df['Year'] = df['Datetime'].dt.year
    df['Month'] = df['Datetime'].dt.month
    df['Day'] = df['Datetime'].dt.day
    df['DayOfWeek'] = df['Datetime'].dt.dayofweek
    df['Hour'] = df['Datetime'].dt.hour
    df['WeekOfYear'] = df['Datetime'].dt.isocalendar().week.astype('Int64')

    df['IsWeekend'] = (
        df['DayOfWeek'].isin([5, 6]).astype(int)
    )

except Exception as e:
    print("Datetime Parsing Error:", e)

# =========================================================
# STANDARDIZE BOOKING STATUS
# =========================================================

if 'Booking Status' in df.columns:

    df['Booking Status'] = (
        df['Booking Status']
        .astype(str)
        .str.strip()
    )

# =========================================================
# CONVERT NUMERIC COLUMNS
# =========================================================

numeric_cols = [
    'Avg VTAT',
    'Avg CTAT',
    'Cancelled Rides by Customer',
    'Cancelled Rides by Driver',
    'Incomplete Rides',
    'Booking Value',
    'Ride Distance',
    'Driver Ratings',
    'Customer Rating'
]

for col in numeric_cols:

    if col in df.columns:

        df[col] = pd.to_numeric(
            df[col],
            errors='coerce'
        )

# =========================================================
# STATUS-AWARE NULL HANDLING
# =========================================================

# ---------------------------------------------------------
# CUSTOMER CANCELLED RIDES
# ---------------------------------------------------------

if 'Booking Status' in df.columns:

    cust_cancel_mask = df['Booking Status'].str.contains(
        'Cancelled by Customer',
        case=False,
        na=False
    )

    if 'Cancelled Rides by Customer' in df.columns:

        df.loc[
            cust_cancel_mask,
            'Cancelled Rides by Customer'
        ] = (
            df.loc[
                cust_cancel_mask,
                'Cancelled Rides by Customer'
            ].fillna(1)
        )

    if 'Cancelled Rides by Driver' in df.columns:

        df.loc[
            cust_cancel_mask,
            'Cancelled Rides by Driver'
        ] = 0

# ---------------------------------------------------------
# DRIVER CANCELLED RIDES
# ---------------------------------------------------------

if 'Booking Status' in df.columns:

    driver_cancel_mask = df['Booking Status'].str.contains(
        'Cancelled by Driver',
        case=False,
        na=False
    )

    if 'Cancelled Rides by Driver' in df.columns:

        df.loc[
            driver_cancel_mask,
            'Cancelled Rides by Driver'
        ] = (
            df.loc[
                driver_cancel_mask,
                'Cancelled Rides by Driver'
            ].fillna(1)
        )

    if 'Cancelled Rides by Customer' in df.columns:

        df.loc[
            driver_cancel_mask,
            'Cancelled Rides by Customer'
        ] = 0

# ---------------------------------------------------------
# INCOMPLETE RIDES
# ---------------------------------------------------------

if 'Booking Status' in df.columns:

    incomplete_mask = df['Booking Status'].str.contains(
        'Incomplete',
        case=False,
        na=False
    )

    if 'Incomplete Rides' in df.columns:

        df.loc[
            incomplete_mask,
            'Incomplete Rides'
        ] = (
            df.loc[
                incomplete_mask,
                'Incomplete Rides'
            ].fillna(1)
        )

# ---------------------------------------------------------
# COMPLETED RIDES
# ---------------------------------------------------------

if 'Booking Status' in df.columns:

    completed_mask = df['Booking Status'].str.contains(
        'Completed',
        case=False,
        na=False
    )

    rating_cols = [
        'Driver Ratings',
        'Customer Rating'
    ]

    for col in rating_cols:

        if col in df.columns:

            median_rating = df.loc[
                completed_mask,
                col
            ].median()

            df.loc[
                completed_mask,
                col
            ] = (
                df.loc[
                    completed_mask,
                    col
                ].fillna(median_rating)
            )

# ---------------------------------------------------------
# BOOKING VALUE & DISTANCE
# ---------------------------------------------------------

active_ride_mask = pd.Series(
    [True] * len(df)
)

if 'Booking Status' in df.columns:

    active_ride_mask = (
        df['Booking Status'].str.contains(
            'Completed',
            case=False,
            na=False
        )
        |
        df['Booking Status'].str.contains(
            'Incomplete',
            case=False,
            na=False
        )
    )

monetary_cols = [
    'Booking Value',
    'Ride Distance'
]

for col in monetary_cols:

    if col in df.columns:

        median_val = df.loc[
            active_ride_mask,
            col
        ].median()

        df.loc[
            active_ride_mask,
            col
        ] = (
            df.loc[
                active_ride_mask,
                col
            ].fillna(median_val)
        )

# ---------------------------------------------------------
# VTAT / CTAT
# ---------------------------------------------------------

wait_cols = [
    'Avg VTAT',
    'Avg CTAT'
]

for col in wait_cols:

    if col in df.columns:

        valid_mask = ~df['Booking Status'].str.contains(
            'Cancelled',
            case=False,
            na=False
        )

        median_wait = df.loc[
            valid_mask,
            col
        ].median()

        df.loc[
            valid_mask,
            col
        ] = (
            df.loc[
                valid_mask,
                col
            ].fillna(median_wait)
        )

# =========================================================
# REASON COLUMNS
# Replace non-applicable NULLs with 'NA'
# =========================================================

reason_cols = [
    'Reason for cancelling by Customer',
    'Driver Cancellation Reason',
    'Incomplete Rides Reason'
]

for col in reason_cols:

    if col in df.columns:

        df[col] = df[col].fillna('NA')

# =========================================================
# OTHER CATEGORICAL COLUMNS
# =========================================================

categorical_cols = [
    'Customer ID',
    'Vehicle Type',
    'Pickup Location',
    'Drop Location',
    'Payment Method'
]

for col in categorical_cols:

    if col in df.columns:

        mode_value = df[col].mode()[0]

        df[col] = df[col].fillna(mode_value)

# =========================================================
# OUTLIER TREATMENT
# IQR CAPPING
# =========================================================

outlier_cols = [
    'Avg VTAT',
    'Avg CTAT',
    'Booking Value',
    'Ride Distance'
]

for col in outlier_cols:

    if col in df.columns:

        Q1 = df[col].quantile(0.25)
        Q3 = df[col].quantile(0.75)

        IQR = Q3 - Q1

        lower_bound = Q1 - 1.5 * IQR
        upper_bound = Q3 + 1.5 * IQR

        df[col] = np.where(
            df[col] < lower_bound,
            lower_bound,
            df[col]
        )

        df[col] = np.where(
            df[col] > upper_bound,
            upper_bound,
            df[col]
        )

# =========================================================
# REMOVE DUPLICATES
# =========================================================

if 'Booking ID' in df.columns:

    before = len(df)

    df = df.drop_duplicates(
        subset=['Booking ID']
    )

    after = len(df)

    print("\nDuplicates Removed:", before - after)

# =========================================================
# FINAL NULL HANDLING
# Replace remaining object-column NULLs with 'NA'
# =========================================================

object_cols = df.select_dtypes(include='object').columns

for col in object_cols:

    df[col] = df[col].fillna('NA')

# =========================================================
# FINAL VALIDATION
# =========================================================

# print("\n==========================")
# print("FINAL DATASET INFO")
# print("==========================")

# print(df.info())

# print("\n==========================")
# print("MISSING VALUES")
# print("==========================")

print(df.isnull().sum())

# =========================================================
# SAVE OUTPUT CSV
# =========================================================

# output_file = "rideBookings_preprocessed1.csv"

# df.to_csv(output_file, index=False)

# print("\nPreprocessed file saved successfully!")

# # =========================================================
# # DOWNLOAD OUTPUT FILE
# # =========================================================

# files.download(output_file)

Dataset Shape: (150000, 21)

Protected Columns:
['Date', 'Time', 'Booking ID', 'Booking Status']

Duplicates Removed: 1233

FINAL DATASET INFO
<class 'pandas.core.frame.DataFrame'>
Index: 148767 entries, 0 to 149999
Data columns (total 29 columns):
 #   Column                             Non-Null Count   Dtype         
---  ------                             --------------   -----         
 0   Date                               148767 non-null  object        
 1   Time                               148767 non-null  object        
 2   Booking ID                         148767 non-null  object        
 3   Booking Status                     148767 non-null  object        
 4   Customer ID                        148767 non-null  object        
 5   Vehicle Type                       148767 non-null  object        
 6   Pickup Location                    148767 non-null  object        
 7   Drop Location                      148767 non-null  object        
 8   Avg VTAT                  

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>